<a href="https://colab.research.google.com/github/TrajanDS/SFT_Agent_POC/blob/main/data_gathering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
import pandas as pd
import time

In [ ]:
CFPB_API_URL = "https://www.consumerfinance.gov/data-research/consumer-complaints/search/api/v1/"

def fetch_cfpb_complaints(
    max_records=1_000,
    batch_size=1_000,
    date_received_min="2024-01-01",
    date_received_max=None,
    has_narrative=True,
):
"""
TODO:
  * Randomize time period being drawn from
  * Get set number of samples for each type
"""


    rows = []
    offset = 0
    headers = {
        "User-Agent": "Mozilla/5.0 (compatible; CFPB-DataFetcher/1.0; https://your-site-or-email)"
    }

    while len(rows) < max_records:
        current_batch = min(batch_size, max_records - len(rows))

        params = {
            "frm": offset,
            "size": current_batch,
            "sort": "created_date_desc",  # or "relevance_desc" etc.
            "no_aggs": "true",
            "date_received_min": date_received_min,
        }
        if date_received_max:
            params["date_received_max"] = date_received_max
        if has_narrative:
            params["has_narrative"] = "true"

        try:
            response = requests.get(CFPB_API_URL, params=params, headers=headers, timeout=60)
            response.raise_for_status()
            data = response.json()

            hits = data.get("hits", {}).get("hits", [])
            if not hits:
                print("No more results returned.")
                break

            # Extract the actual complaint data (_source)
            for hit in hits:
                if "_source" in hit:
                    rows.append(hit["_source"])
                else:
                    rows.append(hit)

            print(f"Fetched {len(hits)} records (total so far: {len(rows)})")

            offset += len(hits)

            # Be polite to the API
            if len(hits) < current_batch:
                break  # last page

            time.sleep(0.5)  # small delay to avoid rate-limiting

        except requests.exceptions.RequestException as e:
            print(f"Error fetching data: {e}")
            break

    print(f"Completed: {len(rows)} records fetched.")
    return rows


# Call API
rows = fetch_cfpb_complaints()

# Form rows from API as dataframe
df = pd.DataFrame(rows)

In [ ]:
df['product'].value_counts()

In [ ]:
import google.generativeai as genai
from google.colab import userdata
from tqdm.notebook import tqdm # For progress bar

# Fetch your API key from Colab Secrets
GOOGLE_API_KEY = userdata.get('rat_name')
genai.configure(api_key=GOOGLE_API_KEY)

Now, let's initialize the Gemini model. We'll use a specific model suitable for text generation and classification.

In [ ]:
# Initialize the Gemini model
gemini_model = genai.GenerativeModel('gemini-1.5-flash-latest')

Next, we'll create a function to generate a prompt for each complaint and send it to Gemini for classification. We'll ask it to identify the `Product` based on the complaint's narrative and other details, and output it in a structured format (JSON) for easy parsing.

In [ ]:
complaint_row = df.iloc[0,:]

In [ ]:
prompt = f"""You are an expert financial complaint analyst. Your task is to classify the product type of the following consumer complaint. Based on the provided details, identify the most appropriate 'product' category from the predefined list.

Predefined Product Categories: {df['product'].unique().tolist()}

Complaint Details:
Date Received: {complaint_row['date_received']}
Issue: {complaint_row['issue']}
Sub-issue: {complaint_row['sub_issue']}
Company: {complaint_row['company']}
Complaint Narrative: {complaint_row['complaint_what_happened']}

Based on these details, what is the product category for this complaint? Please respond with ONLY the product category string. If you cannot determine a specific product, respond with 'Unknown'."""


# Using generate_content for single turn interaction
response = gemini_model.generate_content(prompt)

In [ ]:
def classify_product_with_gemini(complaint_row):
    prompt = f"""You are an expert financial complaint analyst. Your task is to classify the product type of the following consumer complaint. Based on the provided details, identify the most appropriate 'product' category from the predefined list.

Predefined Product Categories: {df['product'].unique().tolist()}

Complaint Details:
Date Received: {complaint_row['date_received']}
Issue: {complaint_row['issue']}
Sub-issue: {complaint_row['sub_issue']}
Company: {complaint_row['company']}
Complaint Narrative: {complaint_row['complaint_what_happened']}

Based on these details, what is the product category for this complaint? Please respond with ONLY the product category string. If you cannot determine a specific product, respond with 'Unknown'."""

    try:
        # Using generate_content for single turn interaction
        response = gemini_model.generate_content(prompt)
        # Accessing response.text assumes the model provides plain text output
        # If the model is fine-tuned to output JSON, you might parse response.text as JSON
        return response.text.strip()
    except Exception as e:
        print(f"Error classifying complaint ID {complaint_row['complaint_id']}: {e}")
        return "Error_Classification"

# Display a sample prompt to ensure it's well-formed
sample_complaint = df.iloc[0]
print("--- Sample Prompt ---")
print(f"Product Categories: {df['product'].unique().tolist()}\n")
print(f"Complaint Details:\nDate Received: {sample_complaint['date_received']}\nIssue: {sample_complaint['issue']}\nSub-issue: {sample_complaint['sub_issue']}\nCompany: {sample_complaint['company']}\nComplaint Narrative: {sample_complaint['complaint_what_happened']}")
print("\n--- Expected Output Format: Product Category Name ---")

Now, let's apply this function to your DataFrame. This might take some time depending on the number of complaints and API rate limits. We'll add a progress bar using `tqdm`.

In [ ]:
tqdm.pandas() # Initialize tqdm for pandas

#df['gemini_classified_product'] = df.progress_apply(classify_product_with_gemini, axis=1)
test = df.head(10).progress_apply(classify_product_with_gemini, axis=1)

In [ ]:
test

Finally, let's compare Gemini's classifications with the original `product` column and display some results.

In [ ]:
# Display the value counts of the Gemini-classified products
print("Gemini Classified Product Value Counts:")
display(df['gemini_classified_product'].value_counts())

# Display a comparison of original vs. Gemini classified for a few rows
print("\nComparison of Original vs. Gemini Classified Products (first 10 rows):")
display(df[['product', 'gemini_classified_product', 'complaint_what_happened']].head(10))

# Optionally, calculate accuracy (if original labels are considered ground truth)
correct_classifications = (df['product'] == df['gemini_classified_product']).sum()
total_classifications = len(df)
accuracy = correct_classifications / total_classifications
print(f"\nClassification Accuracy (vs. original 'product' column): {accuracy:.2f}")